# Visual Tower Baseline
Fine-tuning EfficientNet-B4 with Focal Loss on MVTec AD dataset. Includes GradCAM explainability.

In [ ]:
import torch
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

from backend.app.data.mvtec_dataset import MVTecDataset
from backend.app.models.two_tower import VisualTower, VisualTowerClassifier
from backend.app.training.trainer import AnomalyTrainer
from backend.app.models.gradcam import VisualGradCAM

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Ensure you have the MVTec dataset downloaded at 'data/mvtec' for this to run
try:
    train_dataset = MVTecDataset(root_dir='../data/mvtec', category='bottle', split='train', transform=transform)
    test_dataset = MVTecDataset(root_dir='../data/mvtec', category='bottle', split='test', transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
except FileNotFoundError:
    print('Please download the MVTec AD dataset to the expected folder.')
    train_loader = None

In [ ]:
if train_loader:
    visual_tower = VisualTower(embed_dim=256, freeze_backbone=True)
    model = VisualTowerClassifier(visual_tower)

    trainer = AnomalyTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=test_loader,
        epochs=5,
        lr=1e-4,
        patience=10,
        checkpoint_dir='../checkpoints'
    )

    trainer.train()

In [ ]:
if train_loader:
    model.eval()
    target_layer = model.visual_tower.backbone.conv_head
    cam = VisualGradCAM(model, target_layer)

    # Fetch a sample batch
    data, mask, target = next(iter(test_loader))
    sample_idx = 0
    image_tensor = data[sample_idx:sample_idx+1]

    heatmap = cam.generate_heatmap(image_tensor)
    
    # Denormalize image for visualization
    img_vis = image_tensor.squeeze().permute(1, 2, 0).numpy()
    img_vis = img_vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_vis = np.clip(img_vis, 0, 1)

    overlay = cam.overlay_heatmap(img_vis, heatmap)

    plt.imshow(overlay)
    plt.title(f'GradCAM - Target: {target[sample_idx].item()}')
    plt.show()